In [1]:
import pandas as pd
import re
import itertools

dataset = 'PXD008841'

## Create a full SDRF annotation for dataset PXD008841

We can start with a partial annotation created by Fragpipe. Look here for a full [teaching module on clinical proteomics](https://hds-sandbox.github.io/proteomics-sandbox/TeachingModule/teachingmodule.html) that explains the use of Fragpipe to process the data.

In [2]:
df = pd.read_table('sdrf.tsv')

Alternatively, we can start from scratch if we don't have a partial SDRF. We will make the list of all raw files below. Skip the next two cells if you have the partial annotation from Fragpipe for the full dataset.

In [3]:
pools_and_prefixes = [([1, 2], 'HJOS2U_20140410_TMTpool'), ([3], 'HJOSLO2U_QEHF_20150318_TMT_pool'), ([4], 'HJOSLO2U_QEHF_20150322_TMT_pool'), ([5], 'HJOSLO2U_QEHF_20150329_TMT_pool')]
tmt_channels = sorted(['TMT' + c for c in ['126', '131N'] + [str(m) + i for m, i in itertools.product(range(127, 131), ['N', 'C'])]])
fractions = range(1, 73)
raw_names = []
for pools, prefix in pools_and_prefixes:
    for pool in pools:
        for fr in fractions:
                raw_names.append(f'{prefix}{pool}_300ugIPG37-49_7of15ul_fr{fr:02d}.raw')

In [4]:
df = pd.DataFrame({
    'comment[data file]': [f for f in raw_names for _ in tmt_channels],
    'comment[fraction identifier]': [i for i in itertools.chain.from_iterable(itertools.repeat(tuple(fractions), 5)) for _ in tmt_channels],
    'comment[label]': list(itertools.chain.from_iterable(itertools.repeat(tuple(tmt_channels), len(raw_names)))),
})

Add more required columns. Run the cells below regardless of the starting point.

In [37]:
df['comment[instrument]'] = 'Q Exactive'
df['comment[technical replicate]'] = 1
df['characteristics[organism]'] = 'Homo Sapiens'
df['characteristics[organism part]'] = 'breast'
df['comment[fraction identifier]'] = df['comment[data file]'].str.extract(r'fr(\d+).raw').astype(int)
df['comment[file uri]'] = 'https://storage.jpostdb.org/JPST000265/' + df['comment[data file]']
df['technology type'] = 'proteomic profiling by mass spectrometry'
df['characteristics[ancestry category]'] = 'not available'
df['characteristics[age]'] = 'not available'
df['characteristics[sex]'] = 'female'
df['characteristics[cell type]'] = 'malignant cell'
df['characteristics[biological replicate]'] = 1

Extract the tumor annotations from the supplementary table. Reminder: the supplementary table is available [here](https://www.nature.com/articles/s41467-019-09018-y#Sec15), under **Supplementary Data 1**.

In [5]:
tumor_id = pd.read_excel('41467_2019_9018_MOESM3_ESM.xlsx', sheet_name='Tumor annotations', usecols=['Tumor ID', 'TMT set nr', 'TMT tag', 'PAM50 subtype'], index_col=(1, 2))

In [6]:
tumor_id.index = pd.MultiIndex.from_tuples([(s, str(label)) for s, label in tumor_id.index], names=tumor_id.index.names)

In [7]:
tumor_types = {
    'Basal': 'basal-like breast carcinoma',
    'LumA': 'luminal A breast carcinoma',
    'LumB': 'luminal B breast carcinoma',
    'HER2': 'HER2 Positive Breast Carcinoma',
    'Normal': 'Normal Breast-Like Subtype of Breast Carcinoma'
}

In [8]:
tumor_id.head()

Tumor ID PAM50 subtype
TMT set nr TMT tag                       
1          126      OSL.53E         Basal
           127N     OSL.567          LumA
           127C     OSL.3FF         Basal
           128N     OSL.55F         Basal
           128C     OSL.46A         Basal

In [9]:
pool_str = 'SN=' + ','.join(tumor_id['Tumor ID'].values)

def get_info(row):
    pool = int(re.search(r'pool(\d)', row['comment[data file]']).group(1))
    try:
        sample = tumor_id.loc[(pool, row['comment[label]'][3:]), 'Tumor ID']
        disease = tumor_types[tumor_id.loc[(pool, row['comment[label]'][3:]), 'PAM50 subtype']]
        pooled = 'not pooled'
    except KeyError:
        sample = 'pool'
        disease = 'breast cancer'
        pooled = pool_str
    assay = f"pool {pool}, fraction {row['comment[fraction identifier]']}"
    return sample, assay, pooled, disease

In [10]:
df[['source name', 'assay name', 'characteristics[pooled sample]', 'characteristics[disease]']] = df.apply(get_info, axis=1, result_type='expand')
df['factor value[disease]'] = df['characteristics[disease]']

In [11]:
# sorting key for ordering of columns
def key(colname):
    if colname == 'source name':
        return 0
    if colname[:15] == 'characteristics':
        return 1
    if colname == 'assay name':
        return 2
    if colname == 'technology type':
        return 3
    if colname[:7] == 'comment':
        return 4
    if colname[:12] == 'factor value':
        return 5
    return 6

In [12]:
df = df[sorted(df.columns, key=key)]

In [13]:
df.to_csv(f'{dataset}.sdrf.tsv', sep='\t', index=False)